> **You do not need to run this notebook to reproduce the results.**
> It cleans the raw collected corpus (`data/raw/channel_comments.csv`), which is **not distributed** (excluded for size and YouTube's terms). Skip it: the reproducible pipeline runs from the included labelled data and fixed splits, starting at **notebook 04**. See README §6–§7.

# 02 - Cleaning comments (RQ1)

Prepares the raw corpus for analysis. Reads `data/raw/channel_comments.csv`
and writes `data/processed/comments_clean.csv`.

**Steps**
1. Remove duplicate comment IDs.
2. Decode HTML entities and normalise whitespace.
3. Drop blank comments and exact duplicate texts.
4. Keep comments of five words or more.
5. Keep English-language comments only (`langdetect`).

This reduces the 351,813 raw comments to roughly 280,070 eligible for sampling.

In [ ]:
import pandas as pd
import html
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "channel_comments.csv")
print("loaded:", df.shape)

In [ ]:
clean = df.drop_duplicates("comment_id").reset_index(drop=True)
clean["text_clean"] = (
    clean["text"].map(lambda x: html.unescape(str(x)))
    .str.replace(r"\s+", " ", regex=True).str.strip()
)
before = len(clean)
clean = clean[clean["text_clean"].str.len() > 0]
clean = clean[clean["text_clean"].str.lower() != "nan"]
clean = clean.drop_duplicates(subset="text_clean", keep="first").reset_index(drop=True)
print(f"{before} -> {len(clean)} after removing blanks + duplicate texts")

In [ ]:
clean["n_words"] = clean["text_clean"].str.split().str.len()
before = len(clean)
clean = clean[clean["n_words"] >= 5].reset_index(drop=True)
print(f"{before} -> {len(clean)} after keeping comments with >= 5 words")

In [ ]:
from langdetect import detect, DetectorFactory, LangDetectException
DetectorFactory.seed = 0

def is_english(text):
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False

before = len(clean)
clean["is_en"] = clean["text_clean"].map(is_english)
clean = clean[clean["is_en"]].drop(columns="is_en").reset_index(drop=True)
print(f"{before} -> {len(clean)} after keeping English only")

In [ ]:
clean = clean.drop(columns=[c for c in ["hit_1B","hit_2","hit_any"] if c in clean.columns])
print("final shape:", clean.shape)
print("\nper channel:\n", clean["channel_handle"].value_counts())
print("\nduplicate texts remaining:", clean.duplicated("text_clean").sum())  # should be 0

In [ ]:
out_path = PROJECT_ROOT / "data" / "processed" / "comments_clean.csv"
clean.to_csv(out_path, index=False)
print(f"Saved {len(clean)} clean comments to {out_path}")